# 09 · From the toy to copper

Chapter 09 takes the toy chain you ran in chapters 01–07 and the facts of copper's reference chain, and returns a map between the two, a list of the assumptions the framework makes, and four places where you extend it.

In [1]:
import contextlib
import inspect
import io
import json
import logging
from pathlib import Path

import pandas as pd

from pysuto.tutorial import toy, toy_ext
from pysuto.registry import ConstraintRunRegistry, ImputationRunRegistry

RUNS = Path("runs")
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
logging.disable(logging.INFO)


def quiet():
    return contextlib.redirect_stdout(io.StringIO())


def sidecar(stage, run_id):
    return json.loads((RUNS / "proc" / stage / run_id / "metadata.json").read_text())

The toy is a model of the method, not of the metal: three regions, five years, a planted error and a ground truth you can look up. Copper is the application the method was built for. This chapter is short on new ideas and long on pointers: what is the same, what is different, and where the framework asks you to bring your own code.

## The toy and copper, side by side

The toy's numbers below are read from your run. Copper's are static text, each with its source: a clone of this repository has no copper run, since the feeds that build one are licensed (manuscript §3.4), so nothing here computes them.

In [2]:
base = toy_ext.base_ids(RUNS)
const, opt = sidecar("const", base["const"]), sidecar("opt", base["opt"])
n_year_valuation = const["stage_summary"]["n_edges"] // (len(toy.YEARS) * 2)
size_kb = {k: sum(f.stat().st_size for f in (RUNS / "proc" / k / base[k]).rglob("*") if f.is_file()) / 1e3 for k in ("const", "opt")}
print(f"toy: {const['stage_summary']['n_edges']} edges = {len(toy.YEARS)} years × 2 valuations × {n_year_valuation} edges; "
      f"{const['stage_summary']['n_rows']} constraint rows")
print(f"toy: Stage 4 solved {len(opt['stage_summary']['timesteps'])} years in {opt['stage_summary']['solve_wall_s']:.3f} s; "
      f"the const run is {size_kb['const']:.0f} kB and the opt run {size_kb['opt']:.0f} kB")

toy: 420 edges = 5 years × 2 valuations × 42 edges; 410 constraint rows
toy: Stage 4 solved 5 years in 0.021 s; the const run is 91 kB and the opt run 39 kB


Now the table. The copper column reads the reference chain `opt_20261116_134050_b54f65dc` (sidecars of the run, its const parent `const_20261116_133444_28ef32f2` and its imp grandparent `imp_20261116_133053_ad1191d2`) and the manuscript's Section 3.2 (`docs/manuscript/method.md`).

In [3]:
rows = [
    ("regions", "3 (Montara, Portua, Isla); a closed world", "47, among them `ROW`, the rest of the world, with a pedigree penalty on its edges (ADR 0003; manuscript §3.2.1)"),
    ("years", "5 (2001–2005)", "30 (1995–2024)"),
    ("valuations", "`Gross` (kt) and `Content` (kt Content), declared by the toy itself (#123)", "gross mass (kt) and copper content (kt CC); roles are the framework's, labels the case's (ADR 0034)"),
    ("edges per valuation and year", "42", "14,241"),
    ("edges in all", "420", "854,460 (const sidecar `n_edges`)"),
    ("constraint rows in all", "410", "1,685,688 (const sidecar `n_rows`)"),
    ("Stage 4, all years", f"{opt['stage_summary']['solve_wall_s']:.3f} s", "about 93 s on 8 threads, 30 QPs solved concurrently (opt sidecar `solve_wall_s`; ADR 0039 and #70)"),
    ("disk, one run", f"{size_kb['const']:.0f} kB (const run), {size_kb['opt']:.0f} kB (opt run)", "about 74 MB for the optimiser run, about 0.85 GB for the const run it solved"),
    ("feeds", "three synthetic: `survey`, `bureau`, `customs`; one deliberate error", "three real: ICSG yearbooks (v18), BACI (v33), crude-ore extraction (v3), each behind its own parse layer (manuscript §3.2.2 and Table 1)"),
    ("feed build order", "none: the synthetic feeds are independent", "load-bearing: BACI parse → register gross BACI → ICSG netting against it → re-export feed → BACI feed → trade shares → ICSG feed (ADR 0024, 0027, 0030)"),
    ("hard bands", "none: every row of the toy's QP is an equality (chapter 04)", "the non-negativity of every flow, among others (ADR 0039)"),
    ("Stage 5", "closed-form Gaussian; no active set", "linearised at the active set of the QP: a binding bound pins its edge (ADR 0044)"),
    ("open world", "none", "`ROW` with a pedigree penalty: 0.217 against 0.068 on ICSG (manuscript Table 1)"),
    ("licensed data", "none; the feeds are drawn from a seeded ground truth", "ICSG yearbooks and the S&P ore allocation are licensed and not shared; the parse scripts rebuild the feeds from them (manuscript §3.4)"),
]
pd.DataFrame(rows, columns=["", "toy case", "copper reference chain"]).set_index("")

,toy case,copper reference chain
,,
regions,"3 (Montara, Portua, Isla); a closed world","47, among them `ROW`, the rest of the world, with a pedigree penalty on its edges (ADR 0003; manuscript §3.2.1)"
years,5 (2001–2005),30 (1995–2024)
valuations,"`Gross` (kt) and `Content` (kt Content), declared by the toy itself (#123)","gross mass (kt) and copper content (kt CC); roles are the framework's, labels the case's (ADR 0034)"
edges per valuation and year,42,"14,241"
edges in all,420,"854,460 (const sidecar `n_edges`)"
constraint rows in all,410,"1,685,688 (const sidecar `n_rows`)"
"Stage 4, all years",0.021 s,"about 93 s on 8 threads, 30 QPs solved concurrently (opt sidecar `solve_wall_s`; ADR 0039 and #70)"
"disk, one run","91 kB (const run), 39 kB (opt run)","about 74 MB for the optimiser run, about 0.85 GB for the const run it solved"
feeds,"three synthetic: `survey`, `bureau`, `customs`; one deliberate error","three real: ICSG yearbooks (v18), BACI (v33), crude-ore extraction (v3), each behind its own parse layer (manuscript §3.2.2 and Table 1)"


Three differences deserve a sentence each.

- **The toy has no hard bands, so it has no interior-point iteration** and no tolerance to get wrong (chapter 08, experiment 3). Copper has both, and ADR 0044's active-set Stage 5 exists because of them.
- **The toy's feeds are independent, copper's are not.** The trade shares that weight ICSG's producer fan read the BACI feed, and a stale set of weights once cost 6.2 percentage points of the scale-total objective (ADR 0024, 0027, 0030). The build order is a data dependency that the code checks (a refusal when the input was parsed against another BACI version), not a style.
- **The toy is closed.** Everything exported by one region is imported by another, so every balance can close. In copper, `ROW` absorbs what no listed region reports; that is a modelling choice with a price (ADR 0003).

<details><summary>Going deeper: what has to change to make a new case</summary>

A case is declared in code (`pysuto.tutorial`), not in a configuration file: an accounting-structure spec (ADR 0046), an imputation section and a constraint section whose UPPER_CASE fields are the knobs, and its feeds. The toy is one such declaration in `pysuto/tutorial/toy.py`; copper is another, outside the package, in the repository's `scripts/copper.py`. The stages never mention either: they receive a `CaseSpec` and a data directory (`python -m pysuto run --case toy --data-dir <dir>`).

What does not change with the case is the list of assumptions further down. What does change is the content of the sections: which sectors, which feeds, which parameters CSVs (concentrations, efficiencies, scrap shares). A new metal is mostly a new accounting spec and new feeds; a new region system is a new concordance (chapter 01).
</details>

## The assumptions, in one table

Every chapter from 01 to 06 opens with a box, "What this stage assumes", that cites the framework's commitments by number (`A1` … `A12`, README, "What the framework assumes"). Here they are in one table, each with what the toy does and with the choice copper meets in manuscript Table 2 (`docs/manuscript/method.md` §3.2.7), where Table 2 has one. The column "Chapters" lists the boxes that state the assumption.

In [4]:
A = [
    ("A1", "The prior is Gaussian with a diagonal Σ_a (Eq. 2)", "04, 05", "same", "none: not a choice Table 2 weighs (method §3.1.6)"),
    ("A2", "Feeds combine by inverse variance; every σ composition is one power-mean rule, p = 1.3 (Eq. 1)", "01, 02", "p = 1.3 in every consumer", "size-damped leg σ (γ = 0.5) is built on this rule (ADR 0017)"),
    ("A3", "Hard rows hold exactly; soft rows are priced by σ_c⁻² (Eq. 2)", "03, 04, 05", "hard mass balances and hard stripping relation; soft ratio rows; no scale totals", "scale totals self-anchored, 0.3 kt floor on feed-backed totals only; waste rock fixed by a hard stripping relation (ADR 0006, 0025, 0038)"),
    ("A4", "One QP per period; no row couples two periods", "03, 04", "five QPs, one per year", "thirty QPs, each solved to its numerical floor, lower bounds in the QP (ADR 0039)"),
    ("A5", "No stage before Stage 6 drops σ", "01, 02, 05, 06", "σ carried from Stage 1 to Stage 5; dropped at Stage 6", "same; Stage 6 is the documented exception (PIOT as a derived view, ADR 0037)"),
    ("A6", "A source edge's outgoing weights sum to one; fan-in diversifies, fan-out leaves the CV unchanged", "01", "one fan-in (the Bureau); one two-feed edge", "a record that names one endpoint is fanned by external trade shares, never a uniform split; unweighted fans are not distributed; domestic leg capped at own production; re-exports re-attributed at feed time (ADR 0008, 0018, 0024, 0027)"),
    ("A7", "Pedigree scores map to a CV by Laner's exponential model", "01", "pedigree per feed, set by the toy", "pedigree per feed and a `ROW` penalty (manuscript Table 1; ADR 0003)"),
    ("A8", "An uncovered edge is a flat prior at value 0 with a large σ", "02, 03", "σ = 1e6; flat-prior fills are the last imputer", "no-data anchor at 5 % of the sibling median; extraction chain exempt where an ore-grade relation sets it (ADR 0013)"),
    ("A9", "Imputation fills are write-once and logged", "02", "the imputation log (chapter 02)", "bilateral legs are never time-interpolated; ore grade carried past 2022 (ADR 0026, 0023)"),
    ("A10", "A satellite flow carries a value but sits outside its node's balance", "03, 06", "waste rock", "waste rock a satellite mass flow, 2 × ore, free of copper (ADR 0038)"),
    ("A11", "Stage 6 propagates point estimates only", "06", "same", "PIOT as a derived view under the market-share construct, never persisted (ADR 0007, 0037)"),
    ("A12", "Results are year-local: a period slice is scattered through `cols_t` before any `G @ F`", "04, 05", "same", "each year solved on its own slice, lower bounds enforced in the QP (ADR 0039)"),
]
pd.DataFrame(A, columns=["", "assumption (README)", "Chapters", "the toy", "copper's choice (Table 2)"]).set_index("")

,assumption (README),Chapters,the toy,copper's choice (Table 2)
,,,,
A1,The prior is Gaussian with a diagonal Σ_a (Eq. 2),"04, 05",same,none: not a choice Table 2 weighs (method §3.1.6)
A2,"Feeds combine by inverse variance; every σ composition is one power-mean rule, p = 1.3 (Eq. 1)","01, 02",p = 1.3 in every consumer,size-damped leg σ (γ = 0.5) is built on this rule (ADR 0017)
A3,Hard rows hold exactly; soft rows are priced by σ_c⁻² (Eq. 2),"03, 04, 05",hard mass balances and hard stripping relation; soft ratio rows; no scale totals,"scale totals self-anchored, 0.3 kt floor on feed-backed totals only; waste rock fixed by a hard stripping relation (ADR 0006, 0025, 0038)"
A4,One QP per period; no row couples two periods,"03, 04","five QPs, one per year","thirty QPs, each solved to its numerical floor, lower bounds in the QP (ADR 0039)"
A5,No stage before Stage 6 drops σ,"01, 02, 05, 06",σ carried from Stage 1 to Stage 5; dropped at Stage 6,"same; Stage 6 is the documented exception (PIOT as a derived view, ADR 0037)"
A6,"A source edge's outgoing weights sum to one; fan-in diversifies, fan-out leaves the CV unchanged",01,one fan-in (the Bureau); one two-feed edge,"a record that names one endpoint is fanned by external trade shares, never a uniform split; unweighted fans are not distributed; domestic leg capped at own production; re-exports re-attributed at feed time (ADR 0008, 0018, 0024, 0027)"
A7,Pedigree scores map to a CV by Laner's exponential model,01,"pedigree per feed, set by the toy",pedigree per feed and a `ROW` penalty (manuscript Table 1; ADR 0003)
A8,An uncovered edge is a flat prior at value 0 with a large σ,"02, 03",σ = 1e6; flat-prior fills are the last imputer,no-data anchor at 5 % of the sibling median; extraction chain exempt where an ore-grade relation sets it (ADR 0013)
A9,Imputation fills are write-once and logged,02,the imputation log (chapter 02),"bilateral legs are never time-interpolated; ore grade carried past 2022 (ADR 0026, 0023)"


The table is long on purpose. It is the list of things a second case **cannot** change by passing different data or parameters: if your metal needs a full covariance between edges (A1), or a constraint that couples two years (A4), the answer is a new stage, not a new case.

## Four seams where you extend the framework

Four places take your code. The first three are run below on the toy, from outside the library (`pysuto.tutorial.toy_ext`); the fourth is described, not built.

| seam | you write | the stage that calls it |
| --- | --- | --- |
| imputer | a `BaseImputer` subclass | Stage 2 |
| constraint builder | a `BaseConstraintBuilder` subclass | Stage 3 |
| feed | a synthetic or real feed registered through the Stage 1 API | Stage 1 |
| posterior propagator | a `BaseUncProp` subclass | Stage 5 |

All three working seams run in one chain, Stages 2–4, under the registry label `toy_seams`; your base chain stays the latest run of the toy.

In [5]:
with quiet():
    seams = toy_ext.run_seams(RUNS)
print(json.dumps(seams["runs"]["feed"], indent=1))
print(f"edges the scrap seed wrote: {seams['n_seeded']}")

{
 "survey": "feed_survey_20261123_151850_9449ce2e",
 "bureau": "feed_bureau_20261123_151850_8b4128fb",
 "customs": "feed_customs_20261123_151850_ad6e11a0",
 "ministry": "feed_ministry_20261123_151937_19d0a2f6"
}
edges the scrap seed wrote: 2


### Seam 1 · an imputer: a scrap-share seed

The toy's `Scrap → Smelting` legs are unobserved in the two region-years the Bureau did not publish (chapter 02). The seed writes them at 10 % of the concentrate input the smelter already has, with a σ of 50 % of that: a modelling claim, stated in a few lines. `writer_scope` is the set of edges the imputer may write; the ledger refuses a write outside it, and refuses an overwrite.

In [6]:
print(inspect.getsource(toy_ext.ScrapShareSeed))

class ScrapShareSeed(BaseImputer):
    """Seam 1, a ``BaseImputer``: seed an unwritten ``Scrap → Smelting`` leg (content) at
    ``share`` of the concentrate input its smelter already holds, σ = ``cv`` · seed. The
    ledger enforces no-overwrite and the scope declared here; nothing else is written."""

    def __init__(self, share: float = 0.10, cv: float = 0.5, id: str = "scrap_seed"):
        self.share, self.cv, self.id = share, cv, id

    def prepare(self, acc) -> None:
        em, content = acc.edge_meta, acc.valuation_roles.content
        self._scope = ((em["from_sector"] == "Scrap") & (em["to_sector"] == "Smelting")
                       & (em["valuation"] == content)).to_numpy()
        feed = em[(em["from_sector"] == "Concentrate") & (em["to_sector"] == "Smelting")
                  & (em["valuation"] == content)]
        self._inputs = {k: g["edge_id"].to_numpy()
                        for k, g in feed.groupby(["to_region", "time"])}

    def writer_scope(self, acc) -> n

The seed joins the terminal imputers in front of the `TimeImputer` (so it writes before the time imputer can). The imputation log, the audit trail of chapter 02, shows what it wrote:

In [7]:
log = ImputationRunRegistry(str(RUNS / "proc" / "imp")).load(seams["runs"]["imp"]).imputation_log.as_dataframe()
log[log["imputer_id"] == "scrap_seed"][["edge_id", "method", "source_edges", "coefficients", "f_after", "sigma_after"]]

,edge_id,method,source_edges,coefficients,f_after,sigma_after
267,337,scrap_share_seed,[187],[0.1],0.222,0.111
268,355,scrap_share_seed,"[205, 265]","[0.1, 0.1]",0.417,0.209


Two edges, each with its source edges and its coefficient (`0.1`), value and σ in `kt Content`. Nothing else in the prior changed because of the seed; every other number comes from the cascade of chapter 02.

### Seam 2 · a constraint builder: a soft net-stock row

One soft row per region and year: the net stock change, `Metal → Stock` in `Content`, equals zero at σ = 0.1 kt Content. The builder returns a block of rows, `G`, `c`, σ_c and a `row_meta` frame whose columns carry the row's identity. Read identity from those columns, never by parsing the row's id (GLOSSARY, "Constraint identity").

In [8]:
print(inspect.getsource(toy_ext.NetStockRow))

class NetStockRow(BaseConstraintBuilder):
    """Seam 2, a ``BaseConstraintBuilder``: one soft row per region and year, the net
    stock change ``Metal → Stock`` (content) = 0 at ``sigma`` kt Content. The row's
    ``row_meta`` carries its identity; nothing parses its id."""

    def __init__(self, sigma: float = 0.1):
        self.sigma = sigma

    def build_static(self, acc) -> ConstraintBlock:
        em, content = acc.edge_meta, acc.valuation_roles.content
        hit = em[(em["to_sector"] == "Stock") & (em["valuation"] == content)]
        m, meta = len(hit), pd.DataFrame({c: [None] * len(hit) for c in ROW_META_COLS})
        meta["constraint_id"] = [build_constraint_id("sum", "netstock", r, year=t)
                                 for r, t in zip(hit["to_region"], hit["time"])]
        meta["type"], meta["variant"] = "sum", "netstock"
        meta["is_hard"], meta["sense"] = False, "eq"
        meta["time"], meta["valuation"] = hit["time"].to_numpy(), content
        meta["sect

It joins the toy's builders in Stage 3. The chain solved, so look at what the rows cost: the z of every net-stock row.

In [9]:
const_run = ConstraintRunRegistry(str(RUNS / "proc" / "const")).load(seams["runs"]["const"])
meta = const_run.row_meta
ns = meta[meta["variant"] == "netstock"]
soft = toy.soft_residuals(RUNS, seams["runs"]["opt"])
z = soft[soft["constraint_id"].isin(set(ns["constraint_id"]))].set_index("constraint_id")["z"]
print(f"{len(ns)} net-stock rows ({ns['is_hard'].sum()} hard); largest |z| {z.abs().max():.2f}, median |z| {z.abs().median():.2f}")
z.reindex(z.abs().sort_values(ascending=False).index).head(4).to_frame("z")

15 net-stock rows (0 hard); largest |z| 7.66, median |z| 1.89


,z
constraint_id,
sum_netstock_Portua_2001,7.657
sum_netstock_Portua_2005,-6.452
sum_netstock_Portua_2002,4.251
sum_netstock_Montara_2001,3.627


The rows are cheap to write and not always right. "Net stock change is zero" is a claim about the world, and the solver prices it: Portua's three largest rows have |z| above 4 because the other rows and sources do not support a zero net stock change in Portua. A new builder is a hypothesis; the residual table of chapter 04 is where you learn whether the data agree. The planted leg of chapter 07 is still where it was:

In [10]:
edges = soft[soft["kind"] == "edge"]
base_edges = toy.soft_residuals(RUNS, base["opt"]).query("kind == 'edge'")
print(f"largest edge residual: edge {int(edges.iloc[0]['edge_id'])}, z {edges.iloc[0]['z']:.3f} (base chain: edge {int(base_edges.iloc[0]['edge_id'])}, z {base_edges.iloc[0]['z']:.3f})")

largest edge residual: edge 194, z -4.779 (base chain: edge 194, z -4.779)


### Seam 3 · a fourth feed through the Stage 1 API

A feed is a table plus its concordance maps plus its pedigree; Stage 1 turns that into the framework's root-index form (chapter 01). A new feed is a `SyntheticFeed` here, and a parse layer in copper. The *Ministry* reports the smelters' refined output in exactly the two region-years the Bureau left out, noisy at the Bureau's pedigree.

In [11]:
print(inspect.getsource(toy_ext.fourth_feed))

def fourth_feed(run_root, label: str = "toy_ministry", seed: int = toy.SEED) -> str:
    """Seam 3: a synthetic fourth feed through the Stage 1 API. The *Ministry* reports
    the smelters' refined output (content) in exactly the two region-years the Bureau did
    not publish, noisy at the Bureau's pedigree. Returns its feed run id."""
    truth = toy.truth(seed).set_index(toy._EDGE_KEY)["value"]
    rows = [(t, r, "smelter", "refined",
             float(truth.loc[(t, toy.CONTENT, "Smelting", r, "Metal", r)]))
            for r, t in toy._BUREAU_GAPS]
    raw = pd.DataFrame(rows, columns=["Year", "Region", "Process", "Item", "_truth"]
                       ).assign(Unit=toy._BUREAU_UNIT)
    meta = dataclasses.replace(toy._metadata("bureau"), feed_name="ministry")
    eps = np.random.default_rng([seed, 4]).standard_normal(len(raw))
    cv  = meta.to_pedigree_scores().total_cv()
    raw = raw.assign(Value=raw.pop("_truth") * (1 + cv * eps))
    feed = toy.SyntheticFeed("ministry", ra

Nothing else is needed: the feed is registered under its own label, and Stage 2 collects it with the other three. The first output cell of this section showed its run id beside the three feeds of chapter 01. The count below shows what it adds to the aggregate.

In [12]:
covered = {name: sidecar("imp", run)["stage_summary"]["n_covered_before"] for name, run in (("base", base["imp"]), ("seams", seams["runs"]["imp"]))}
print(f"edges covered by the feeds before imputation: {covered['base']} on the base chain, {covered['seams']} with the Ministry")

edges covered by the feeds before imputation: 93 on the base chain, 95 with the Ministry


<details><summary>Going deeper: what each seam guarantees and what it does not</summary>

- **Imputer:** the ledger guarantees write-once and scope; it does not check that your fill is sensible. The σ you write is the σ the QP uses (Eq. 2), after the Lp floor of `_conditioning.py`.
- **Builder:** the engine rejects rows that couple two years (A4); it does not check that the row is consistent with the others. An inconsistent hard row makes the QP infeasible and Stage 4 raises (never a silent fallback); an inconsistent soft row shows up as a z.
- **Feed:** Stage 1 enforces the weight guard (A6) and the pedigree-to-CV map (A7); the feed's *content* is yours. A defect in the feed is the case of chapter 08, experiment 1.
</details>

### Seam 4 · a posterior propagator (described, not built)

Stage 5 is an abstract class with one method, `BaseUncProp.compute(opt_run_id, alpha)`, that returns an `UncPropResult`: for every edge and year, the marginal σ and the interval bounds, and optionally posterior draws. `KalmanUncProp` is the closed-form concrete of chapter 05, a QR of the whitened system per year, linearised at the active set (ADR 0044). The README names the siblings that slot in beside it: a Monte Carlo propagator that samples the truncated posterior instead of linearising it, and a bounds-aware one. Each reads the same run tree (the opt run's sidecar names its const run) and returns the same container; Stage 6 and the scoreboard of chapter 07 do not notice which one ran. No example is built here: a correct propagator is a piece of numerical analysis, and the active-set linearisation is the one the framework ships.

## Where to read next

- **README**: the architecture and the module reference. Start with "Architecture", then the stage you want to change; "What the framework assumes" holds the list A1–A12.
- **GLOSSARY.md**: one definition per word the tutorial used (retention, period slice, satellite flow, hard and soft).
- **`docs/adr/`**: why each choice was made, with the run that measured it. The four that chapter 08 leans on are 0019, 0035 (σ cannot repair a contradiction), 0039 (the termination rule) and 0044 (the active-set posterior).
- **`docs/manuscript/method.md`**: the same pipeline on copper, with the equations numbered as here (Eq. 1–3) and Table 2 of the modelling choices.
- **`docs/rep/`**: the analyses behind the ADRs, one per run.

Run your own metal next: chapter 10 writes a minimal case outside the package and runs it, and the toy's `pysuto/tutorial/toy.py` is the longer model to copy from. Keep the synthetic feeds until the real ones parse.

## Where this run came from

The seams chain wrote its own runs. The optimiser run's sidecar shows its chain:

In [13]:
print(json.dumps(sidecar("opt", seams["runs"]["opt"])["parents"], indent=2))
print(json.dumps(sidecar("imp", seams["runs"]["imp"])["parents"]["upstream"], indent=2))

{
  "inputs": {},
  "upstream": {
    "const_run_id": "const_20261123_151942_38d17fc7"
  }
}
{
  "feed_run_ids": [
    "feed_survey_20261123_151850_9449ce2e",
    "feed_bureau_20261123_151850_8b4128fb",
    "feed_customs_20261123_151850_ad6e11a0",
    "feed_ministry_20261123_151937_19d0a2f6"
  ]
}


The imputation run names four feed runs, the three of chapter 01 and the Ministry's. Chapter 10 is the next step: it writes a case of your own in a file and runs it through the same command.